# CareLoop Clinics — No-Show Guard
## Part 4 · Turn call-centre notes into structured reasons (Tasks 4.1–4.4)

**Inputs:** `data/followup_notes.csv`, `appointments_clean.csv` · **Output:** `notes_analysed.csv`

### Before you run
1. Get a free key in **Google AI Studio** (no card).
2. The key is typed at run time with `input()`. **Never** paste it into a code cell.
3. After the key cell runs, clear its output (⋮ → Clear output).
4. Before downloading, use **Edit → Find** to search the whole notebook for the first few
   characters of your key. It must not appear anywhere.

API budget: 1 call (4.1) + 9 (4.2, N004 is blank) + 4 (4.3) = **14 calls**, well under 25.

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import json
import time
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

# Use the Flash model from your Session 22.1 notebook. If it has been retired,
# pick the current Flash model listed in Google AI Studio.
MODEL_NAME = "gemini-3-flash-preview"

# Session 22.1 sets thinking_level="minimal". If your model rejects thinking_config,
# set this to False: everything else (JSON mode, schema, Pydantic check) stays the same.
USE_THINKING_CONFIG = True

ALLOWED_CAUSES = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
                  "illness_or_emergency", "work_or_schedule", "other"]

API_CALLS = 0          # counts every request sent to Gemini in this notebook

In [ ]:
API_KEY = input("Paste your key: ")          # typed at run time, never saved in the notebook
client = genai.Client(api_key=API_KEY)
print("Client ready.")                       # clear this cell's output after running

## Task 4.1 — One validated call
**CRAFT prompt**
- **C**ontext: CareLoop outpatient clinics; the note is written after phoning a patient who missed a visit.
- **R**ole: operations analyst who classifies missed-appointment reasons.
- **A**ction: pick exactly one `root_cause` from the allowed list, give confidence, summary, action.
- **T**one: factual and neutral; no guessing beyond the note.
- **F**ormat: JSON with exactly four fields (enforced by JSON mode + schema).

In [ ]:
SYSTEM_TEXT = ("You analyse short call-centre notes for an outpatient clinic chain. "
               "Reply only with a JSON object that matches the given schema.")

SCHEMA = {
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

def build_craft_prompt(note_text):
    return f"""CONTEXT: CareLoop Clinics runs eight outpatient clinics in India. After a patient
misses an appointment, the call centre phones them and writes a short note. Notes may be in
English, Hindi or Hinglish.

ROLE: You are an operations analyst who classifies why patients missed appointments.

ACTION: Read the note and choose exactly ONE root_cause from this list:
- forgot_or_no_reminder : patient forgot, or says no reminder/SMS/call was received
- distance_or_transport : clinic too far, no transport, travel problems
- cost_or_fee           : fee too high, cannot afford, money problems
- illness_or_emergency  : patient (or family member) ill, medical or family emergency
- work_or_schedule      : work, leave refused, shift or timing clash
- other                 : anything else, including when the patient could not be reached
                          or the reason is unclear
If the note does not state a reason clearly, answer "other" rather than guess.
Also give a confidence between 0 and 1, a summary of at most 25 words, and one concrete
recommended_action for the clinic.

TONE: Factual, neutral and brief. Do not invent details that are not in the note.

FORMAT: JSON with exactly these keys: root_cause (one value from the list, written exactly),
confidence (number 0-1), summary (string, max 25 words), recommended_action (string).

NOTE: \"\"\"{note_text}\"\"\""""

def make_config():
    kwargs = dict(system_instruction=SYSTEM_TEXT, temperature=0, max_output_tokens=1000,
                  response_mime_type="application/json", response_schema=SCHEMA)
    if USE_THINKING_CONFIG:
        kwargs["thinking_config"] = types.ThinkingConfig(thinking_level="minimal")
    return types.GenerateContentConfig(**kwargs)

def call_gemini(prompt):
    """Send one prompt; return the raw reply text exactly as received."""
    global API_CALLS
    API_CALLS += 1
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt, config=make_config())
    return response.text

def analyse_note(note_text):
    """Classify one note. Returns (parsed_dict, raw_text)."""
    raw = call_gemini(build_craft_prompt(note_text))
    return json.loads(raw), raw

In [ ]:
notes = pd.read_csv("data/followup_notes.csv")
n001 = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]
print("N001:", n001)

data_n001, raw_n001 = analyse_note(n001)
print(json.dumps(data_n001, indent=2))
print("Fields:", sorted(data_n001.keys()))

## Task 4.2 — Validate and process the first 10 notes
### (a) Pydantic model + allowed-list check

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def check_allowed(insight):
    if insight.root_cause not in ALLOWED_CAUSES:
        raise ValueError(f"root_cause '{insight.root_cause}' is not in the allowed list")
    return insight

### (b)–(d) Loop, validate, keep every reply unrepaired, save

In [ ]:
FIELDS = ["root_cause", "confidence", "summary", "recommended_action"]
records = []

for _, row in notes.head(10).iterrows():
    note_id, appt_id, note_text = row["note_id"], row["appointment_id"], row["note_text"]
    rec = {"note_id": note_id, "appointment_id": appt_id}

    # Input validation: NaN first (a float has no .strip()), then empty / spaces only
    if pd.isna(note_text) or str(note_text).strip() == "":
        rec.update(root_cause="other", confidence=0.0, summary="No note recorded",
                   recommended_action="Re-call patient", validation_status="skipped_blank",
                   raw_response="")
        records.append(rec)
        print(f"{note_id}: blank note -> skipped, no API call")
        continue

    raw = None
    try:
        time.sleep(13)                              # free-tier rate limit
        raw = call_gemini(build_craft_prompt(note_text))
        data = json.loads(raw)
        for f in FIELDS:                            # copy whatever fields came back, as returned
            rec[f] = data.get(f) if isinstance(data, dict) else None
        check_allowed(FollowupInsight(**data))
        rec["validation_status"] = "valid"
    except (ValidationError, ValueError, TypeError) as e:
        rec["validation_status"] = "invalid"
        print(f"{note_id}: INVALID -> {str(e).splitlines()[0]}")
    except Exception as e:                          # API / network errors must not stop the loop
        rec["validation_status"] = "invalid"
        print(f"{note_id}: API error -> {type(e).__name__}: {e}")
    rec["raw_response"] = raw if raw is not None else ""
    records.append(rec)
    print(f"{note_id}: {rec.get('root_cause')} ({rec['validation_status']})")

COLS = ["note_id", "appointment_id", "root_cause", "confidence", "summary",
        "recommended_action", "validation_status", "raw_response"]
analysed = pd.DataFrame(records).reindex(columns=COLS)
analysed.to_csv("notes_analysed.csv", index=False)
print("\nSaved notes_analysed.csv:", analysed.shape)
print("API calls so far:", API_CALLS)
analysed[["note_id", "root_cause", "confidence", "validation_status"]]

In [ ]:
EXPECTED = {"N001": "forgot_or_no_reminder", "N002": "distance_or_transport", "N003": "cost_or_fee",
            "N004": "other", "N005": "illness_or_emergency", "N006": "work_or_schedule",
            "N007": "forgot_or_no_reminder", "N008": "distance_or_transport", "N009": "other",
            "N010": "work_or_schedule"}
check = analysed[["note_id", "root_cause"]].copy()
check["expected"] = check["note_id"].map(EXPECTED)
check["match"] = check["root_cause"] == check["expected"]
non_blank = check[check["note_id"] != "N004"]
print(f"Agreement on the 9 non-blank notes: {non_blank['match'].sum()} / 9")
check

## Task 4.3 — Break it on purpose
### (a) Two deliberately bad replies (no API calls)

In [ ]:
bad_replies = {
    "confidence = 1.7": {"root_cause": "cost_or_fee", "confidence": 1.7,
                         "summary": "Patient said the fee was too high.",
                         "recommended_action": "Offer the follow-up discount."},
    "missing recommended_action": {"root_cause": "cost_or_fee", "confidence": 0.8,
                                   "summary": "Patient said the fee was too high."},
}
for label, reply in bad_replies.items():
    try:
        FollowupInsight(**reply)
        print(f"{label}: unexpectedly passed")
    except ValidationError as e:
        print(f"--- {label} ---")
        for err in e.errors():
            print(f"  field={err['loc']}  type={err['type']}  msg={err['msg']}")

### (b) Prompt V1 vs V2 on N007 (Hinglish) and N009 (ambiguous)
Same model, schema, system instruction, JSON mode and temperature=0; only the prompt differs.

In [ ]:
def build_v1_prompt(note_text):
    return f"Classify this note: {note_text}"

results = []
for note_id in ["N007", "N009"]:
    text = notes.loc[notes["note_id"] == note_id, "note_text"].iloc[0]
    for version, builder in [("V1", build_v1_prompt), ("V2", build_craft_prompt)]:
        time.sleep(13)
        raw = None
        try:
            raw = call_gemini(builder(text))
            cause = json.loads(raw).get("root_cause")
        except Exception as e:
            cause = f"ERROR: {type(e).__name__}"
        results.append({"note": note_id, "prompt": version, "root_cause": cause,
                        "in_allowed_list": cause in ALLOWED_CAUSES})

prompt_table = pd.DataFrame(results)
prompt_table

**Which prompt to deploy:** V2. Without the allowed list, V1 has no fixed vocabulary, so its
labels are free text that fails the allowed-list check (or must be mapped by hand). V2 names
the six values, defines each, and tells the model to answer `other` when the note gives no
clear reason (N009: the patient could not be reached), so its output passes validation and
does not invent a cause.

### (c) Total API calls

In [ ]:
print("Total API calls in Tasks 4.1–4.3:", API_CALLS)
print("  4.1: 1 | 4.2: 9 (N004 blank, skipped) | 4.3: 4  -> expected 14")
assert API_CALLS < 25

## Task 4.4 — Combine with the appointment data

In [ ]:
appts = pd.read_csv("appointments_clean.csv")
merged = analysed.merge(appts, on="appointment_id", how="left")
print("Merged rows:", len(merged))
pd.crosstab(merged["root_cause"], merged["reminder_sent"], margins=True)

In [ ]:
print("Notes whose appointment had reminder_sent = No:",
      (merged["reminder_sent"] == "No").sum(), "of", len(merged))
forgot = merged[merged["root_cause"] == "forgot_or_no_reminder"]
print("forgot_or_no_reminder notes:", forgot["note_id"].tolist(),
      "-> reminder_sent:", forgot["reminder_sent"].tolist())

**Final check before download:** clear the key cell's output, then Edit → Find for the first
characters of your key. Nothing should match.